# Flight Price SQL + MLlib Starter

Notebook inicial para validar o ambiente PySpark, executar consultas SQL e treinar modelos basicos com `pyspark.ml`.

Diretrizes deste notebook:
- leitura do dataset via HDFS, com fallback local apenas em modo local;
- preparacao, validacao e exploracao usando `spark.sql(...)`;
- modelagem com MLlib sobre uma base preparada em SQL;
- comparacao entre modelos com e sem `base_fare`;
- split temporal para reduzir vazamento entre treino e teste.


In [1]:
import os
from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor, LinearRegression


In [2]:
APP_NAME = "flight-price-sql-mllib"
SPARK_MASTER_URL = os.environ.get("SPARK_MASTER", "spark://spark-master:7077")
HDFS_PATH = "hdfs://namenode:9000/data/itineraries.csv"
LOCAL_PATH = "/data/itineraries.csv"
LOAD_SAMPLE_RATIO = 0.001  # reduza para 0.001 se quiser iterar ainda mais rapido
DEV_SAMPLE_RATIO = 1.0

RAW_SCHEMA = """
legId STRING,
searchDate STRING,
flightDate STRING,
startingAirport STRING,
destinationAirport STRING,
fareBasisCode STRING,
travelDuration STRING,
elapsedDays STRING,
isBasicEconomy STRING,
isRefundable STRING,
isNonStop STRING,
baseFare STRING,
totalFare STRING,
seatsRemaining STRING,
totalTravelDistance STRING,
segmentsDepartureTimeEpochSeconds STRING,
segmentsDepartureTimeRaw STRING,
segmentsArrivalTimeEpochSeconds STRING,
segmentsArrivalTimeRaw STRING,
segmentsArrivalAirportCode STRING,
segmentsDepartureAirportCode STRING,
segmentsAirlineName STRING,
segmentsAirlineCode STRING,
segmentsEquipmentDescription STRING,
segmentsDurationInSeconds STRING,
segmentsDistance STRING,
segmentsCabinCode STRING
"""


def get_or_create_spark(app_name: str = APP_NAME) -> SparkSession:
    spark_session = (
        SparkSession.builder
        .appName(app_name)
        .master(SPARK_MASTER_URL)
        .config("spark.sql.shuffle.partitions", "16")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
        .config("spark.sql.repl.eagerEval.enabled", "true")
        .config("spark.sql.repl.eagerEval.maxNumRows", "20")
        .config("spark.sql.repl.eagerEval.truncate", "80")
        .getOrCreate()
    )
    spark_session.sparkContext.setLogLevel("WARN")
    return spark_session


def resolve_candidate_paths(spark_session: SparkSession) -> list[str]:
    master = spark_session.sparkContext.master
    if master.startswith("local"):
        return [HDFS_PATH, LOCAL_PATH]
    return [HDFS_PATH]


def load_flights_csv(spark_session: SparkSession, candidate_paths: list[str]):
    last_error = None
    print(f"Tentando carregar dataset pelos caminhos: {candidate_paths}")
    for path in candidate_paths:
        try:
            df = (
                spark_session.read
                .option("header", True)
                .schema(RAW_SCHEMA)
                .csv(path)
            )
            print(f"Dataset registrado com sucesso: {path}")
            return df, path
        except Exception as exc:
            print(f"Falha ao carregar {path}: {exc}")
            last_error = exc
    raise RuntimeError("Nao foi possivel carregar o dataset em nenhum caminho") from last_error


def apply_load_sample(df, ratio: float):
    if ratio >= 1.0:
        print("Usando dataset completo apos a leitura.")
        return df
    print(f"Aplicando amostra de carga com ratio={ratio} para acelerar as iteracoes.")
    return df.sample(withReplacement=False, fraction=ratio, seed=42)


def run_sql(query: str, preview_rows: int = 20):
    df = spark.sql(query)
    display(df.limit(preview_rows).toPandas())
    return df


In [3]:
spark = get_or_create_spark()
candidate_paths = resolve_candidate_paths(spark)
raw_input_df, source_path = load_flights_csv(spark, candidate_paths)
raw_df = apply_load_sample(raw_input_df, LOAD_SAMPLE_RATIO)
raw_df.createOrReplaceTempView("flights_raw")

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Source path:", source_path)
print("Load sample ratio:", LOAD_SAMPLE_RATIO)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/01 22:44:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Tentando carregar dataset pelos caminhos: ['hdfs://namenode:9000/data/itineraries.csv']
Dataset registrado com sucesso: hdfs://namenode:9000/data/itineraries.csv
Aplicando amostra de carga com ratio=0.001 para acelerar as iteracoes.
Spark version: 3.5.3
Spark master: spark://spark-master:7077
Source path: hdfs://namenode:9000/data/itineraries.csv
Load sample ratio: 0.001


26/05/01 22:44:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [4]:
run_sql("SELECT * FROM flights_raw LIMIT 5")


,legId,searchDate,flightDate,startingAirport,destinationAirport,fareBasisCode,travelDuration,elapsedDays,isBasicEconomy,isRefundable,...,segmentsArrivalTimeEpochSeconds,segmentsArrivalTimeRaw,segmentsArrivalAirportCode,segmentsDepartureAirportCode,segmentsAirlineName,segmentsAirlineCode,segmentsEquipmentDescription,segmentsDurationInSeconds,segmentsDistance,segmentsCabinCode
0,002ae21f5d6f9f47057dde59f97f4821,2022-04-16,2022-04-17,PHL,LAX,QA0NA0MQ,PT8H20M,0,False,False,...,1650212640||1650224100,2022-04-17T10:24:00.000-06:00||2022-04-17T12:3...,SLC||LAX,PHL||SLC,Delta||Delta,DL||DL,Airbus A321||Boeing 737-900,18540||7260,1922||590,coach||coach
1,7105aa1fd1ab485475217565fddf32f2,2022-04-16,2022-04-17,PHL,LGA,KA0NX0MQ,PT5H24M,0,False,False,...,1650197100||1650209040,2022-04-17T08:05:00.000-04:00||2022-04-17T11:2...,ATL||LGA,PHL||ATL,Delta||Delta,DL||DL,Boeing 737-900||Airbus A320,7500||7800,667||762,coach||coach
2,4fe3b785aae2645cca25f1c294e3527b,2022-04-16,2022-04-17,SFO,BOS,VAA0KKEN,PT7H44M,0,False,False,...,1650206160||1650224640,2022-04-17T08:36:00.000-06:00||2022-04-17T15:4...,DEN||BOS,SFO||DEN,United||United,UA||UA,Boeing 757-300||Boeing 737 MAX 9,9360||14040,954||1763,coach||coach
3,43b01ee8997c75adf16ac5fbaa28dd02,2022-04-16,2022-04-18,ATL,BOS,V0AJZNN1,PT6H8M,0,False,False,...,1650294960||1650310020,2022-04-18T11:16:00.000-04:00||2022-04-18T15:2...,MIA||BOS,ATL||MIA,American Airlines||American Airlines,AA||AA,Airbus A319||,7020||11820,596||1260,coach||coach
4,3d1bd94d5ade1dc3c8bdbe83e44842f0,2022-04-16,2022-04-18,BOS,DTW,V0AJZNN1,PT13H37M,0,False,False,...,1650294000||1650327660||1650337620,2022-04-18T11:00:00.000-04:00||2022-04-18T19:2...,ROC||ORD||DTW,BOS||ROC||ORD,American Airlines||American Airlines||American...,AA||AA||AA,Embraer RJ145||Embraer 175||Embraer 175,5400||7140||5220,343||525||240,coach||coach||coach


legId,searchDate,flightDate,startingAirport,destinationAirport,fareBasisCode,travelDuration,elapsedDays,isBasicEconomy,isRefundable,isNonStop,baseFare,totalFare,seatsRemaining,totalTravelDistance,segmentsDepartureTimeEpochSeconds,segmentsDepartureTimeRaw,segmentsArrivalTimeEpochSeconds,segmentsArrivalTimeRaw,segmentsArrivalAirportCode,segmentsDepartureAirportCode,segmentsAirlineName,segmentsAirlineCode,segmentsEquipmentDescription,segmentsDurationInSeconds,segmentsDistance,segmentsCabinCode
002ae21f5d6f9f47057dde59f97f4821,2022-04-16,2022-04-17,PHL,LAX,QA0NA0MQ,PT8H20M,0,False,False,False,322.79,370.60,6,2512,1650194100||1650216840,2022-04-17T07:15:00.000-04:00||2022-04-17T11:34:00.000-06:00,1650212640||1650224100,2022-04-17T10:24:00.000-06:00||2022-04-17T12:35:00.000-07:00,SLC||LAX,PHL||SLC,Delta||Delta,DL||DL,Airbus A321||Boeing 737-900,18540||7260,1922||590,coach||coach
7105aa1fd1ab485475217565fddf32f2,2022-04-16,2022-04-17,PHL,LGA,KA0NX0MQ,PT5H24M,0,False,False,False,388.83,441.59,9,1429,1650189600||1650201240,2022-04-17T06:00:00.000-04:00||2022-04-17T09:14:00.000-04:00,1650197100||1650209040,2022-04-17T08:05:00.000-04:00||2022-04-17T11:24:00.000-04:00,ATL||LGA,PHL||ATL,Delta||Delta,DL||DL,Boeing 737-900||Airbus A320,7500||7800,667||762,coach||coach
4fe3b785aae2645cca25f1c294e3527b,2022-04-16,2022-04-17,SFO,BOS,VAA0KKEN,PT7H44M,0,False,False,False,322.79,370.60,7,2717,1650196800||1650210600,2022-04-17T05:00:00.000-07:00||2022-04-17T09:50:00.000-06:00,1650206160||1650224640,2022-04-17T08:36:00.000-06:00||2022-04-17T15:44:00.000-04:00,DEN||BOS,SFO||DEN,United||United,UA||UA,Boeing 757-300||Boeing 737 MAX 9,9360||14040,954||1763,coach||coach
43b01ee8997c75adf16ac5fbaa28dd02,2022-04-16,2022-04-18,ATL,BOS,V0AJZNN1,PT6H8M,0,False,False,False,213.02,252.60,5,1856,1650287940||1650298200,2022-04-18T09:19:00.000-04:00||2022-04-18T12:10:00.000-04:00,1650294960||1650310020,2022-04-18T11:16:00.000-04:00||2022-04-18T15:27:00.000-04:00,MIA||BOS,ATL||MIA,American Airlines||American Airlines,AA||AA,Airbus A319||,7020||11820,596||1260,coach||coach
3d1bd94d5ade1dc3c8bdbe83e44842f0,2022-04-16,2022-04-18,BOS,DTW,V0AJZNN1,PT13H37M,0,False,False,False,315.35,372.70,1,1108,1650288600||1650320520||1650332400,2022-04-18T09:30:00.000-04:00||2022-04-18T18:22:00.000-04:00||2022-04-18T20:4...,1650294000||1650327660||1650337620,2022-04-18T11:00:00.000-04:00||2022-04-18T19:21:00.000-05:00||2022-04-18T23:0...,ROC||ORD||DTW,BOS||ROC||ORD,American Airlines||American Airlines||American Airlines,AA||AA||AA,Embraer RJ145||Embraer 175||Embraer 175,5400||7140||5220,343||525||240,coach||coach||coach


In [5]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW flights_clean AS
WITH prepared AS (
    SELECT
        legId AS leg_id,
        TO_DATE(searchDate) AS search_date,
        TO_DATE(flightDate) AS flight_date,
        startingAirport AS starting_airport,
        destinationAirport AS destination_airport,
        fareBasisCode AS fare_basis_code,
        travelDuration AS travel_duration_iso,
        CAST(elapsedDays AS INT) AS overnight_days,
        DATEDIFF(TO_DATE(flightDate), TO_DATE(searchDate)) AS days_until_flight,
        CASE WHEN LOWER(CAST(isBasicEconomy AS STRING)) = 'true' THEN 1 ELSE 0 END AS is_basic_economy,
        CASE WHEN LOWER(CAST(isRefundable AS STRING)) = 'true' THEN 1 ELSE 0 END AS is_refundable,
        CASE WHEN LOWER(CAST(isNonStop AS STRING)) = 'true' THEN 1 ELSE 0 END AS is_non_stop,
        CAST(baseFare AS DOUBLE) AS base_fare,
        CAST(totalFare AS DOUBLE) AS total_fare,
        CAST(seatsRemaining AS INT) AS seats_remaining,
        CAST(totalTravelDistance AS DOUBLE) AS total_travel_distance,
        (
            CAST(CASE WHEN REGEXP_EXTRACT(travelDuration, '([0-9]+)D', 1) = '' THEN '0' ELSE REGEXP_EXTRACT(travelDuration, '([0-9]+)D', 1) END AS INT) * 1440
            + CAST(CASE WHEN REGEXP_EXTRACT(travelDuration, '([0-9]+)H', 1) = '' THEN '0' ELSE REGEXP_EXTRACT(travelDuration, '([0-9]+)H', 1) END AS INT) * 60
            + CAST(CASE WHEN REGEXP_EXTRACT(travelDuration, '([0-9]+)M', 1) = '' THEN '0' ELSE REGEXP_EXTRACT(travelDuration, '([0-9]+)M', 1) END AS INT)
        ) AS travel_duration_minutes,
        CASE
            WHEN segmentsDepartureAirportCode IS NULL OR TRIM(segmentsDepartureAirportCode) = '' THEN 0
            ELSE CAST(((LENGTH(segmentsDepartureAirportCode) - LENGTH(REPLACE(segmentsDepartureAirportCode, '||', ''))) / 2) + 1 AS INT)
        END AS segment_count,
        DAYOFWEEK(TO_DATE(flightDate)) AS flight_day_of_week,
        MONTH(TO_DATE(flightDate)) AS flight_month,
        CONCAT(startingAirport, '-', destinationAirport) AS route
    FROM flights_raw
    WHERE totalFare IS NOT NULL
)
SELECT
    *,
    GREATEST(segment_count - 1, 0) AS stop_count
FROM prepared
""")

if DEV_SAMPLE_RATIO < 1.0:
    spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW flights_dev AS
    SELECT *
    FROM flights_clean
    WHERE rand(42) <= {DEV_SAMPLE_RATIO}
    """)
else:
    spark.sql("CREATE OR REPLACE TEMP VIEW flights_dev AS SELECT * FROM flights_clean")


In [6]:
run_sql("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN days_until_flight < 0 THEN 1 ELSE 0 END) AS invalid_days_until_flight,
    SUM(CASE WHEN segment_count <= 0 THEN 1 ELSE 0 END) AS invalid_segment_count,
    SUM(CASE WHEN travel_duration_minutes <= 0 THEN 1 ELSE 0 END) AS invalid_travel_duration,
    MIN(total_fare) AS min_total_fare,
    MAX(total_fare) AS max_total_fare,
    ROUND(AVG(total_fare), 2) AS avg_total_fare,
    ROUND(AVG(travel_duration_minutes), 2) AS avg_duration_minutes,
    ROUND(AVG(days_until_flight), 2) AS avg_days_until_flight,
    ROUND(AVG(overnight_days), 2) AS avg_overnight_days
FROM flights_dev
""")


,total_rows,invalid_days_until_flight,invalid_segment_count,invalid_travel_duration,min_total_fare,max_total_fare,avg_total_fare,avg_duration_minutes,avg_days_until_flight,avg_overnight_days
0,82394,0,0,0,23.97,3314.1,339.67,427.5,26.89,0.15


total_rows,invalid_days_until_flight,invalid_segment_count,invalid_travel_duration,min_total_fare,max_total_fare,avg_total_fare,avg_duration_minutes,avg_days_until_flight,avg_overnight_days
82394,0,0,0,23.97,3314.1,339.67,427.5,26.89,0.15


In [7]:
run_sql("""
SELECT
    segment_count,
    stop_count,
    COUNT(*) AS total_voos,
    ROUND(AVG(total_fare), 2) AS avg_total_fare,
    ROUND(AVG(travel_duration_minutes), 2) AS avg_duration_minutes
FROM flights_dev
GROUP BY segment_count, stop_count
ORDER BY segment_count, stop_count
LIMIT 20
""")


,segment_count,stop_count,total_voos,avg_total_fare,avg_duration_minutes
0,1,0,22054,250.89,181.06
1,2,1,52516,345.77,479.78
2,3,2,7642,546.63,773.16
3,4,3,182,645.89,689.87


segment_count,stop_count,total_voos,avg_total_fare,avg_duration_minutes
1,0,22054,250.89,181.06
2,1,52516,345.77,479.78
3,2,7642,546.63,773.16
4,3,182,645.89,689.87


In [8]:
run_sql("""
SELECT
    route,
    travel_duration_iso,
    travel_duration_minutes,
    days_until_flight,
    overnight_days,
    segment_count,
    stop_count,
    total_fare
FROM flights_dev
WHERE travel_duration_minutes <= 0
   OR segment_count <= 0
   OR days_until_flight < 0
ORDER BY total_fare DESC
LIMIT 20
""")


,route,travel_duration_iso,travel_duration_minutes,days_until_flight,overnight_days,segment_count,stop_count,total_fare


route,travel_duration_iso,travel_duration_minutes,days_until_flight,overnight_days,segment_count,stop_count,total_fare


In [9]:
run_sql("""
SELECT
    route,
    COUNT(*) AS total_voos,
    ROUND(AVG(total_fare), 2) AS avg_total_fare,
    ROUND(AVG(base_fare), 2) AS avg_base_fare
FROM flights_dev
GROUP BY route
ORDER BY avg_total_fare DESC
LIMIT 20
""")


,route,total_voos,avg_total_fare,avg_base_fare
0,LGA-OAK,404,661.07,586.93
1,OAK-LGA,350,652.55,579.17
2,OAK-CLT,217,646.01,571.46
3,MIA-OAK,137,645.06,565.51
4,OAK-BOS,259,644.95,572.30
5,OAK-MIA,173,643.67,567.75
6,IAD-OAK,186,640.30,568.35
7,BOS-OAK,287,638.86,565.88
8,CLT-OAK,203,620.30,546.66
9,PHL-OAK,231,617.31,544.34


route,total_voos,avg_total_fare,avg_base_fare
LGA-OAK,404,661.07,586.93
OAK-LGA,350,652.55,579.17
OAK-CLT,217,646.01,571.46
MIA-OAK,137,645.06,565.51
OAK-BOS,259,644.95,572.3
OAK-MIA,173,643.67,567.75
IAD-OAK,186,640.3,568.35
BOS-OAK,287,638.86,565.88
CLT-OAK,203,620.3,546.66
PHL-OAK,231,617.31,544.34


In [ ]:
run_sql("""
SELECT
    is_non_stop,
    COUNT(*) AS total_voos,
    ROUND(AVG(total_fare), 2) AS avg_total_fare,
    ROUND(AVG(days_until_flight), 2) AS avg_days_until_flight,
    ROUND(AVG(overnight_days), 2) AS avg_overnight_days
FROM flights_dev
GROUP BY is_non_stop
ORDER BY is_non_stop DESC
""")


,is_non_stop,total_voos,avg_total_fare,avg_days_until_flight,avg_overnight_days
0,1,22054,250.89,27.27,0.05
1,0,60340,372.12,26.75,0.19


[Stage 39:=======>                                               (30 + 2) / 232]

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW flights_ml_base AS
SELECT
    total_fare,
    base_fare,
    days_until_flight,
    overnight_days,
    seats_remaining,
    COALESCE(total_travel_distance, 0.0) AS total_travel_distance,
    travel_duration_minutes,
    segment_count,
    stop_count,
    flight_day_of_week,
    flight_month,
    is_basic_economy,
    is_refundable,
    is_non_stop,
    route,
    flight_date
FROM flights_dev
WHERE total_fare IS NOT NULL
  AND base_fare IS NOT NULL
  AND days_until_flight IS NOT NULL
  AND overnight_days IS NOT NULL
  AND seats_remaining IS NOT NULL
  AND travel_duration_minutes IS NOT NULL
  AND segment_count BETWEEN 1 AND 8
  AND travel_duration_minutes BETWEEN 30 AND 4320
  AND days_until_flight BETWEEN 0 AND 365
""")

run_sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(flight_date) AS min_flight_date,
    MAX(flight_date) AS max_flight_date,
    ROUND(AVG(total_fare), 2) AS avg_total_fare,
    ROUND(AVG(days_until_flight), 2) AS avg_days_until_flight,
    ROUND(AVG(segment_count), 2) AS avg_segment_count
FROM flights_ml_base
""")

cutoff_unix = spark.sql("SELECT percentile_approx(unix_timestamp(flight_date), 0.8) AS cutoff_unix FROM flights_ml_base").first()["cutoff_unix"]

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW flights_train AS
SELECT *
FROM flights_ml_base
WHERE unix_timestamp(flight_date) <= {cutoff_unix}
""")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW flights_test AS
SELECT *
FROM flights_ml_base
WHERE unix_timestamp(flight_date) > {cutoff_unix}
""")

train_df = spark.sql("SELECT * FROM flights_train")
test_df = spark.sql("SELECT * FROM flights_test")
train_df.cache()
test_df.cache()

run_sql("""
SELECT 'train' AS split_name, COUNT(*) AS total_rows, MIN(flight_date) AS min_flight_date, MAX(flight_date) AS max_flight_date FROM flights_train
UNION ALL
SELECT 'test' AS split_name, COUNT(*) AS total_rows, MIN(flight_date) AS min_flight_date, MAX(flight_date) AS max_flight_date FROM flights_test
""")

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())


In [ ]:
COMMON_FEATURE_COLS = [
    "days_until_flight",
    "overnight_days",
    "seats_remaining",
    "total_travel_distance",
    "travel_duration_minutes",
    "segment_count",
    "stop_count",
    "flight_day_of_week",
    "flight_month",
    "is_basic_economy",
    "is_refundable",
    "is_non_stop"
]

FEATURE_SETS = {
    "with_base_fare": ["base_fare"] + COMMON_FEATURE_COLS,
    "without_base_fare": COMMON_FEATURE_COLS
}

regression_evaluators = {
    "rmse": RegressionEvaluator(labelCol="total_fare", predictionCol="prediction", metricName="rmse"),
    "mae": RegressionEvaluator(labelCol="total_fare", predictionCol="prediction", metricName="mae"),
    "r2": RegressionEvaluator(labelCol="total_fare", predictionCol="prediction", metricName="r2")
}

train_count = train_df.count()
test_count = test_df.count()


def build_pipeline(feature_cols, estimator):
    route_indexer = StringIndexer(inputCol="route", outputCol="route_index", handleInvalid="keep")
    route_encoder = OneHotEncoder(inputCols=["route_index"], outputCols=["route_ohe"])
    assembler = VectorAssembler(inputCols=feature_cols + ["route_ohe"], outputCol="features")
    return Pipeline(stages=[route_indexer, route_encoder, assembler, estimator])


def train_and_evaluate(model_name: str, estimator, feature_set_name: str, feature_cols):
    pipeline = build_pipeline(feature_cols, estimator)
    fitted_pipeline = pipeline.fit(train_df)
    predictions = fitted_pipeline.transform(test_df)
    metrics = {
        "model": model_name,
        "feature_set": feature_set_name,
        "uses_base_fare": 1 if "base_fare" in feature_cols else 0,
        "rmse": regression_evaluators["rmse"].evaluate(predictions),
        "mae": regression_evaluators["mae"].evaluate(predictions),
        "r2": regression_evaluators["r2"].evaluate(predictions),
        "train_rows": train_count,
        "test_rows": test_count
    }
    return fitted_pipeline, predictions, metrics


In [ ]:
model_runs = {}
results = []

for feature_set_name, feature_cols in FEATURE_SETS.items():
    linear_regression = LinearRegression(
        featuresCol="features",
        labelCol="total_fare",
        predictionCol="prediction",
        maxIter=20,
        regParam=0.1,
        elasticNetParam=0.0
    )

    decision_tree = DecisionTreeRegressor(
        featuresCol="features",
        labelCol="total_fare",
        predictionCol="prediction",
        maxDepth=8,
        minInstancesPerNode=50
    )

    for base_model_name, estimator in [
        ("linear_regression", linear_regression),
        ("decision_tree", decision_tree)
    ]:
        run_name = f"{base_model_name}__{feature_set_name}"
        fitted_pipeline, predictions, metrics = train_and_evaluate(run_name, estimator, feature_set_name, feature_cols)
        model_runs[run_name] = {
            "pipeline": fitted_pipeline,
            "predictions": predictions,
            "metrics": metrics
        }
        results.append(metrics)

metrics_df = spark.createDataFrame(results)
display(metrics_df.orderBy("rmse").toPandas())


In [ ]:
best_run_name = metrics_df.orderBy("rmse").first()["model"]
best_predictions = model_runs[best_run_name]["predictions"]
best_predictions.createOrReplaceTempView("best_predictions")

print("Best run:", best_run_name)

run_sql("""
SELECT
    route,
    total_fare,
    ROUND(prediction, 2) AS prediction,
    ROUND(ABS(total_fare - prediction), 2) AS absolute_error,
    days_until_flight,
    overnight_days,
    travel_duration_minutes,
    segment_count,
    stop_count,
    is_non_stop
FROM best_predictions
ORDER BY absolute_error DESC
LIMIT 20
""")


## Proximos passos

- reduzir `LOAD_SAMPLE_RATIO` para validacoes rapidas ou aumentar gradualmente para aproximar do dataset completo;
- persistir `flights_clean` e `flights_ml_base` em Parquet no HDFS para acelerar novas execucoes;
- testar `RandomForestRegressor` e `GBTRegressor` como proximos baselines;
- adicionar novas features a partir dos campos por segmento, mas sempre com validacoes SQL logo apos o parsing.


In [ ]:
print("Notebook pronto para exploracao interativa no Jupyter Lab.")
